# 📝 하이브리드 검색 과제 LV1(기초)

교안 01의 하이브리드 검색을 한 단계씩 연습합니다. 한국어 백과 문서 5개(`lv1_docs.json`)로 BM25와 Dense 검색기를 만들고, `EnsembleRetriever`로 합친 뒤 같은 후보에서 가중치를 비교합니다.

- 1~3번: 한국어 토큰화, BM25 검색기, BM25 점수 읽기
- 4~5번: Dense 검색기, `EnsembleRetriever` 하이브리드 검색기
- 6번: 같은 후보로 BM25·Dense·두 가중치의 Recall@2 비교
- 7~8번: RRF와 BM25의 원리를 말로 설명

문서가 5개뿐이라 모든 검색기는 `k=2`로 2개씩 가져옵니다. `.env`의 OpenAI 키가 필요합니다. 임베딩만 요청하고(문서 5개와 검색 질문, 자가채점의 확인 검색) GPT는 호출하지 않습니다.

**풀이 방법**: 준비 셀과 `[제공 코드]` 셀을 위에서부터 실행한 뒤, 문항 순서대로 답안 셀을 채우고 바로 아래 `# [자가채점]` 셀로 확인하세요. 앞 문항의 변수를 다음 문항에서 이어 씁니다. 검색 순위는 고정 정답이 아니므로 자가채점은 처리 순서와 원문 대응만 확인합니다. 결과가 맞는지는 출력된 원문을 읽어 판단하세요. 자가채점이 검색을 다시 해 대조하는 문항은 순위가 거의 같은 문서 때문에 드물게 실패할 수 있으니, 그때는 답안 셀부터 다시 실행하세요.


## 준비와 데이터 살펴보기

노트북이 있는 폴더에서 새 커널로 시작하고 준비 셀을 위에서부터 실행하세요. 경로(`material_dir`·`data_dir`·`output_dir`)와 `read_json`·`save_json`은 앞 단원과 같습니다. 첫 셀에서 `langchain-community` 유지보수 종료를 알리는 경고가 한 번 보일 수 있습니다. `BM25Retriever`를 이 패키지에서 가져오기 때문이며 실행에는 문제가 없습니다.


In [ ]:
# 문서·검색기·모델에 필요한 라이브러리를 가져옵니다.
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv
from kiwipiepy import Kiwi
from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings


In [ ]:
# 실행할 노트북 폴더를 기준으로 입력 data와 생성 결과 output을 구분합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


임베딩은 `text-embedding-3-large`의 768차원입니다. `check_embedding_ctx_length=False`는 자동 길이 검사·분할을 끕니다. API 키는 `.env`에서 읽습니다.


In [ ]:
# 현재 작업 폴더부터 상위로 .env를 찾아 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))

# 문서와 질문을 같은 모델·차원으로 바꿔야 같은 인덱스에서 거리를 비교할 수 있습니다.
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=768,
    check_embedding_ctx_length=False,  # 자동 길이 검사·분할을 끄고 준비한 짧은 문서를 보냅니다.
)
print("모델 연결 설정 완료. 임베딩은 적재·검색 셀에서 요청합니다.")


절 하나가 이미 짧은 발췌라서 다시 나누지 않고 레코드 하나를 검색 단위 하나로 씁니다. 이 단위를 고정한 채 검색 방식과 질문을 바꿔 결과를 비교합니다. `make_documents`는 원문 ID·제목·출처에 필터용 메타데이터를 더해 `Document`를 만듭니다.


In [ ]:
def make_documents(records):
    """원문 기록의 본문·출처와 검색 조건을 LangChain Document로 바꿉니다."""
    # id는 저장소가, metadata의 source_id는 RRF가 원문을 구별할 때 읽습니다.
    # 두 위치에 같은 원문 ID를 사용하고 출처·필터 메타데이터를 담습니다.
    return [Document(
        id=record["doc_id"],
        page_content=record["text"],
        metadata={"source_id": record["doc_id"], "title": record["title"],
                  "url": record["url"], "source": record["source"], **record["metadata"]},
    ) for record in records]


In [ ]:
# 전체 목록을 먼저 보고 질문에 필요한 필터 필드를 확인합니다.
records = read_json("lv1_docs.json")
display(pd.DataFrame(records)[["doc_id", "title", "metadata"]])
print(records[0]["text"])
documents = make_documents(records)


In [ ]:
def show_results(documents):
    """앞 5개 결과의 원문 ID·메타데이터·본문을 모든 열과 함께 보여 줍니다."""
    # 빈 결과는 필터를 바꾸지 않고 그대로 알립니다.
    if not documents:
        print("조건에 맞는 검색 결과가 없습니다.")
        return
    # 합집합·조건 목록에서도 쓰므로 번호를 관련성 순위라고 부르지 않습니다.
    # 열 순서를 고정해야 표를 나란히 비교할 수 있습니다. url은 길어서 뺍니다.
    rows = [{"display_order": order, "source_id": doc.metadata["source_id"], "title": doc.metadata["title"],
             **{key: doc.metadata[key] for key in sorted(doc.metadata) if key not in {"source_id", "title", "url"}},
             "text": doc.page_content} for order, doc in enumerate(documents, start=1)]
    with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
        display(pd.DataFrame(rows).head())


아래 `[제공 코드]` 셀은 실행만 하세요. 교안 01과 같은 토큰화 함수 `kiwi_tokenize`, 벡터 저장소 `vector_store`, Recall 계산 함수 `source_recall`을 준비합니다.


In [ ]:
# [제공 코드]
# 분석기는 한 번 준비해 문서와 질문 양쪽에 같은 방식으로 적용합니다.
kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자를 소문자 토큰 목록으로 돌려줍니다."""
    # PDF 표기 통일: 재택･원격근무 → 재택·원격근무
    text = text.replace("･", "·")
    # N 계열은 명사, SL은 외국어, SN은 숫자입니다. 조사는 제외합니다.
    # lower는 영문 대소문자를 통일합니다. 기호가 중요한 제품 코드는 별도로 살펴봅니다.
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


In [ ]:
# [제공 코드]
# 같은 이름의 수업용 메모리 컬렉션만 비웁니다. 재실행하면 문서를 다시 임베딩합니다.
vector_store = Chroma(collection_name="day47_lv1_wiki", embedding_function=embedding_model)
vector_store.reset_collection()
# make_documents가 넣은 Document.id가 저장소 ID가 되므로 원문 ID가 검색 결과까지 그대로 이어집니다.
added_ids = vector_store.add_documents(documents)
print("day47_lv1_wiki 적재 수:", len(added_ids))


In [ ]:
# [제공 코드]
def source_recall(documents, expected_ids):
    """정답 문서 ID 중 검색된 ID의 비율을 반환합니다."""
    expected = set(expected_ids)
    retrieved_ids = {doc.metadata["source_id"] for doc in documents}
    # 분모는 검색 결과 수가 아니라 필요한 원문 수입니다.
    return len(retrieved_ids & expected) / len(expected)


## 1. 공백 분리와 형태소 토큰을 비교합니다

**배경**: 한국어는 조사가 붙어 공백으로 나누면 ‘데이터베이스를’과 ‘데이터베이스’가 다른 토큰이 됩니다. BM25에 넣을 토큰을 먼저 확인합니다.

**요구사항**:

- **`question`**: 문자열 `"관계형 데이터베이스를 설명해 주세요"`를 저장하세요.
- **`space_tokens`**: `question`을 공백으로 나눈 문자열 목록입니다(`split` 사용).
- **`query_tokens`**: `question`을 제공된 `kiwi_tokenize`에 넣은 문자열 목록입니다.
- **출력**: 두 목록을 한 줄씩 출력해 조사가 어떻게 처리됐는지 비교하세요.

**예시**: `space_tokens`에는 `데이터베이스를`처럼 조사가 붙은 토큰이 남고, `query_tokens`에는 `데이터베이스`처럼 조사가 빠진 명사가 남습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 같은 질문을 두 방식으로 나누어 결과를 나란히 봅니다.

세부구현:
1. question에 질문 문자열을 저장합니다.
2. 문자열의 split으로 공백 분리 목록을 만듭니다.
3. kiwi_tokenize로 형태소 토큰 목록을 만듭니다.
4. 두 목록을 출력해 비교합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert question == '관계형 데이터베이스를 설명해 주세요', '지문의 질문 문장을 그대로 question에 저장하세요.'
assert space_tokens == question.split(), 'space_tokens에는 question을 공백으로 나눈 목록(split 결과)을 담으세요.'
assert isinstance(query_tokens, list) and query_tokens == kiwi_tokenize(question), 'query_tokens에는 kiwi_tokenize(question)의 반환 목록을 그대로 담으세요.'
assert '데이터베이스' in query_tokens and '데이터베이스를' not in query_tokens, 'kiwi_tokenize를 쓰면 조사가 빠진 데이터베이스 토큰이 나와야 합니다.'
print("✅ 1번 통과!")


## 2. BM25 검색기를 만들고 검색합니다

**배경**: ‘GeoJSON’처럼 정확한 용어로 찾을 때는 같은 토큰의 일치를 점수에 반영하는 BM25가 쓸모 있습니다.

**요구사항**:

- **`bm25`**: 전체 `documents`로 만든 `BM25Retriever`입니다. `preprocess_func=kiwi_tokenize`, `bm25_params={"k1": 1.5, "b": 0.75}`, `k=2`를 지정하세요.
- **`bm25_results`**: `bm25`로 문자열 `"GeoJSON"`을 검색한 `Document` 목록입니다. `show_results`로 원문을 읽으세요.

**예시**: `bm25_results`는 2개입니다. `k`는 반환할 최대 문서 수이고, `k1`·`b`는 점수 계산 설정입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색기를 만드는 단계와 질문을 넣는 단계를 나눕니다.

세부구현:
1. BM25Retriever.from_documents에 documents를 넣고 preprocess_func, bm25_params, k를 키워드 인자로 줍니다.
2. 만든 검색기의 invoke에 질문 문자열을 넣습니다.
3. show_results로 결과 원문을 확인합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert isinstance(bm25, BM25Retriever) and len(bm25.docs) == len(documents), 'bm25는 전체 documents로 만든 BM25Retriever여야 합니다.'
assert bm25.k == 2, 'bm25의 k(반환 개수)를 2로 설정하세요.'
assert bm25.preprocess_func is kiwi_tokenize, 'preprocess_func에 kiwi_tokenize를 전달하세요.'
assert bm25.vectorizer.k1 == 1.5 and bm25.vectorizer.b == 0.75, 'bm25_params로 k1=1.5, b=0.75를 전달하세요.'
assert [doc.metadata['source_id'] for doc in bm25_results] == [doc.metadata['source_id'] for doc in bm25.invoke('GeoJSON')], "bm25_results에는 bm25로 'GeoJSON'을 검색한 결과를 담으세요."
print("✅ 2번 통과!")


## 3. BM25 점수를 원문 ID와 함께 읽습니다

**배경**: BM25는 최소 점수로 문서를 거르지 않아서, 점수가 0인 문서로도 `k`개를 채웁니다. 결과에 들어 있다고 근거가 있는 것은 아니므로 점수를 함께 확인합니다.

**요구사항**:

- **`bm25_scores`**: `bm25.vectorizer.get_scores`에 `kiwi_tokenize("GeoJSON")`을 넣어 받은 점수 목록입니다. `get_scores`는 전처리를 하지 않으므로 질문을 직접 토큰화해서 넣습니다.
- **`score_by_id`**: `documents`와 `bm25_scores`를 같은 순서로 짝지어 `{원문 ID: 점수}`로 담은 딕셔너리입니다. 원문 ID는 각 문서의 `metadata["source_id"]`이고, 점수는 `float`로 바꿔 넣으세요.
- **출력**: `score_by_id`와 2번 `bm25_results`의 원문 ID를 함께 출력하세요.

**예시**: `score_by_id`의 키는 원문 ID 5개입니다. ‘GeoJSON’ 토큰이 있는 `wiki4`만 점수가 0보다 크고 나머지는 0인데, `bm25_results`에는 2개가 들어 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 점수 목록은 documents와 같은 순서로 나오므로 같은 위치끼리 짝짓습니다.

세부구현:
1. 질문을 kiwi_tokenize로 토큰화해 get_scores에 넣습니다.
2. zip으로 documents와 bm25_scores를 묶어 원문 ID와 점수의 딕셔너리를 만듭니다.
3. 점수 딕셔너리와 검색 결과 ID를 출력해 점수 0인 문서가 결과에 있는지 봅니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert len(bm25_scores) == len(documents), 'get_scores는 문서마다 점수 하나를 돌려줍니다. kiwi_tokenize("GeoJSON")을 넣었는지 확인하세요.'
assert isinstance(score_by_id, dict) and set(score_by_id) == {doc.metadata['source_id'] for doc in documents}, 'score_by_id의 키는 documents의 원문 ID 5개입니다.'
assert all(type(value) is float for value in score_by_id.values()), '점수는 float로 바꿔 넣으세요.'
assert all(score_by_id[doc.metadata['source_id']] == float(score) for doc, score in zip(documents, bm25_scores)), 'documents와 bm25_scores를 같은 위치끼리 짝지으세요.'
assert score_by_id['wiki4'] > 0, "'GeoJSON' 토큰이 있는 wiki4의 점수가 0입니다. get_scores에 kiwi_tokenize로 토큰화한 질문을 넣었는지 확인하세요."
print("✅ 3번 통과!")


## 4. Dense 검색기를 만들고 BM25와 비교합니다

**배경**: Dense는 질문과 문서를 임베딩 벡터로 바꿔 가까운 문서를 찾습니다. 같은 질문과 같은 반환 개수로 BM25와 결과를 비교합니다.

**요구사항**:

- **`dense`**: 제공된 `vector_store`의 `as_retriever`로 만든 검색기입니다. `search_kwargs={"k": 2}`로 2개를 반환하게 하세요.
- **`dense_question`**: 문자열 `"표 모양으로 정보를 정리해 두고 SQL로 꺼내 보는 시스템"`을 저장하세요. 5번에서도 씁니다.
- **`dense_results`**, **`bm25_compare_results`**: `dense`와 2번 `bm25`로 `dense_question`을 각각 검색한 `Document` 목록입니다.
- **`common_ids`**: 두 결과에 모두 들어 있는 원문 ID의 집합(set)입니다.

**예시**: 두 결과는 각각 2개이고, `common_ids`는 0~2개입니다. `show_results`로 두 결과의 원문을 읽고, 질문과 낱말만 겹치는 문서가 어느 쪽에 들어왔는지 보세요.

<details><summary>힌트</summary>

```text
접근방법:
- 질문과 반환 개수를 같게 두고 검색 방식만 바꿉니다.

세부구현:
1. vector_store.as_retriever에 search_kwargs로 k를 지정합니다.
2. 같은 질문을 dense와 bm25에 각각 넣습니다.
3. 두 결과의 source_id 집합을 만들고 & 연산자로 교집합을 구합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert dense.search_kwargs.get('k') == 2, 'dense는 search_kwargs={"k": 2}로 만드세요.'
assert dense_question == '표 모양으로 정보를 정리해 두고 SQL로 꺼내 보는 시스템', '지문의 질문 문장을 그대로 dense_question에 저장하세요.'
assert isinstance(dense_results, list) and [doc.metadata['source_id'] for doc in dense_results] == [doc.metadata['source_id'] for doc in dense.invoke(dense_question)], 'dense_results에는 dense로 dense_question을 검색한 결과를 담으세요.'
assert [doc.metadata['source_id'] for doc in bm25_compare_results] == [doc.metadata['source_id'] for doc in bm25.invoke(dense_question)], 'bm25_compare_results에는 2번 bm25로 dense_question을 검색한 결과를 담으세요.'
dense_ids = [doc.metadata['source_id'] for doc in dense_results]
bm25_ids = [doc.metadata['source_id'] for doc in bm25_compare_results]
assert isinstance(common_ids, set), 'common_ids는 원문 ID의 집합(set)입니다.'
assert all(source_id in dense_ids and source_id in bm25_ids for source_id in common_ids), 'common_ids에는 두 결과에 모두 있는 원문 ID만 담으세요.'
assert all(source_id in common_ids for source_id in dense_ids if source_id in bm25_ids), '두 결과에 모두 있는 원문 ID가 common_ids에서 빠졌습니다.'
print("✅ 4번 통과!")


## 5. EnsembleRetriever로 하이브리드 검색기를 만듭니다

**배경**: BM25 점수와 Dense 거리는 척도가 달라 그대로 더할 수 없습니다. `EnsembleRetriever`는 두 결과의 등수로 가중 RRF 점수를 매겨 하나의 목록으로 합칩니다.

**요구사항**:

- **`hybrid`**: `retrievers=[bm25, dense]`, `weights=[0.3, 0.7]`, `c=60`, `id_key="source_id"`로 만든 `EnsembleRetriever`입니다. 가중치 순서는 검색기 순서와 같습니다.
- **`hybrid_results`**: `hybrid`로 4번 `dense_question`을 검색한 전체 결과 목록입니다.
- **`hybrid_top`**: `hybrid_results`의 앞 2개입니다. 최종 개수는 합친 뒤에 자릅니다.

**예시**: `hybrid_results`는 두 검색 결과의 합집합이라 2~4개이고 같은 원문 ID가 두 번 나오지 않습니다. `hybrid_top`은 2개입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 두 검색기를 가중치와 함께 묶은 뒤, 합친 결과에서 최종 개수만 자릅니다.

세부구현:
1. EnsembleRetriever에 retrievers, weights, c, id_key를 지정합니다.
2. invoke로 dense_question을 검색해 전체 결과를 저장합니다.
3. 슬라이싱으로 앞 2개를 hybrid_top에 담습니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert isinstance(hybrid, EnsembleRetriever), 'hybrid는 EnsembleRetriever로 만드세요.'
assert hybrid.retrievers[0] is bm25 and hybrid.retrievers[1] is dense, 'retrievers는 [bm25, dense] 순서로 넣으세요.'
assert hybrid.weights == [0.3, 0.7], 'weights는 [0.3, 0.7]입니다. 순서는 retrievers와 같습니다.'
assert hybrid.c == 60 and hybrid.id_key == 'source_id', "c=60, id_key='source_id'를 지정하세요."
hybrid_ids = [doc.metadata['source_id'] for doc in hybrid_results]
assert 2 <= len(hybrid_ids) <= 4 and len(hybrid_ids) == len(set(hybrid_ids)), 'hybrid_results에는 자르지 않은 전체 결과를 담으세요(2~4개, 원문 ID 중복 없음).'
assert hybrid_ids == [doc.metadata['source_id'] for doc in hybrid.invoke(dense_question)], 'hybrid_results에는 hybrid로 4번의 dense_question을 검색한 결과를 자르지 않고 모두 담으세요.'
assert hybrid_top == hybrid_results[:2], 'hybrid_top에는 hybrid_results의 앞 2개를 담으세요.'
print("✅ 5번 통과!")


## 6. 같은 후보로 검색 방식과 가중치를 비교합니다

**배경**: 가중치를 바꾸면 최종 순서가 달라집니다. 공정하게 비교하려면 같은 질문의 **같은 후보**에 두 가중치를 적용하고, 최종 2개에 필요한 원문이 몇 개 들어왔는지(Recall@2)를 봅니다.

**요구사항**:

- **`compare_question`**: 문자열 `"코미디 프로그램에서 이름을 딴 언어와 행과 열로 자료를 저장하는 시스템을 알려 주세요"`를 저장하세요.
- **`expected_ids`**: 이 질문의 정답 원문 ID 집합 `{"wiki0", "wiki1"}`입니다(파이썬과 관계형 데이터베이스).
- **`candidates`**: `[bm25로 검색한 결과, dense로 검색한 결과]` 순서의 목록입니다. 각 검색은 한 번만 합니다.
- **`hybrid_even`**: 5번과 같은 설정에서 `weights`만 `[0.5, 0.5]`로 바꾼 `EnsembleRetriever`입니다.
- **`method_results`**: 방식 이름을 키로, 최종 2개 `Document` 목록을 값으로 담은 딕셔너리입니다. 키는 이 순서로 넣으세요.
  - `"BM25"`, `"Dense"`: `candidates`의 첫째·둘째 목록 그대로(이미 2개씩)
  - `"Hybrid 0.3/0.7"`, `"Hybrid 0.5/0.5"`: `hybrid`, `hybrid_even`의 `weighted_reciprocal_rank`에 `candidates`를 넣은 결과의 앞 2개
- **`recall_by_method`**: `method_results`의 키마다 제공된 `source_recall`로 구한 Recall@2를 담은 딕셔너리입니다.

**예시**: `recall_by_method`의 값은 0, 0.5, 1 가운데 하나입니다. 두 가중치의 Hybrid 결과가 서로 다른지, 어느 방식이 두 정답을 모두 찾았는지 확인하세요. Dense 순위는 임베딩 모델에 따라 달라질 수 있습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 검색은 한 번만 하고, 같은 후보를 두 가중치로 다시 합칩니다. Recall은 제공 함수가 계산합니다.

세부구현:
1. bm25와 dense로 compare_question을 한 번씩 검색해 candidates 목록에 순서대로 담습니다.
2. hybrid_even을 만듭니다.
3. weighted_reciprocal_rank에 candidates를 넣어 두 Hybrid의 순위를 받고, 네 방식 모두 앞 2개를 method_results에 담습니다.
4. 딕셔너리 컴프리헨션으로 방식마다 source_recall을 호출합니다.
```

</details>


In [ ]:
# 여기에 코드를 작성하세요


In [ ]:
# [자가채점]
assert compare_question == '코미디 프로그램에서 이름을 딴 언어와 행과 열로 자료를 저장하는 시스템을 알려 주세요', '지문의 질문 문장을 그대로 compare_question에 저장하세요.'
assert expected_ids == {'wiki0', 'wiki1'}, 'expected_ids는 {"wiki0", "wiki1"}입니다.'
assert isinstance(candidates, list) and len(candidates) == 2, 'candidates는 [BM25 결과, Dense 결과] 두 목록을 담은 목록입니다.'
assert [doc.metadata['source_id'] for doc in candidates[0]] == [doc.metadata['source_id'] for doc in bm25.invoke(compare_question)], 'candidates의 첫 번째는 bm25로 compare_question을 검색한 결과입니다.'
assert [doc.metadata['source_id'] for doc in candidates[1]] == [doc.metadata['source_id'] for doc in dense.invoke(compare_question)], 'candidates의 두 번째는 dense로 compare_question을 검색한 결과입니다.'
assert hybrid_even.weights == [0.5, 0.5] and hybrid_even.retrievers[0] is bm25 and hybrid_even.retrievers[1] is dense and hybrid_even.id_key == 'source_id', 'hybrid_even은 5번과 같은 설정에서 weights만 [0.5, 0.5]로 만드세요.'
assert list(method_results) == ['BM25', 'Dense', 'Hybrid 0.3/0.7', 'Hybrid 0.5/0.5'], 'method_results의 키 이름과 순서를 지문대로 맞추세요.'
assert all(len(results) == 2 for results in method_results.values()), '네 방식 모두 최종 2개만 담으세요.'
assert method_results['BM25'] == candidates[0][:2] and method_results['Dense'] == candidates[1][:2], 'BM25·Dense는 candidates의 각 목록 앞 2개입니다. 다시 검색하지 마세요.'
assert [doc.metadata['source_id'] for doc in method_results['Hybrid 0.3/0.7']] == [doc.metadata['source_id'] for doc in hybrid.weighted_reciprocal_rank(candidates)[:2]], 'Hybrid 0.3/0.7에는 hybrid의 RRF로 candidates를 합친 결과의 앞 2개를 담으세요.'
assert [doc.metadata['source_id'] for doc in method_results['Hybrid 0.5/0.5']] == [doc.metadata['source_id'] for doc in hybrid_even.weighted_reciprocal_rank(candidates)[:2]], 'Hybrid 0.5/0.5에는 hybrid_even의 RRF로 candidates를 합친 결과의 앞 2개를 담으세요.'
assert list(recall_by_method) == list(method_results) and all(recall_by_method[name] == source_recall(method_results[name], expected_ids) for name in method_results), 'recall_by_method에는 방식마다 최종 2개와 expected_ids로 source_recall을 호출한 값을 담으세요.'
print("✅ 6번 통과!")


## 7. RRF가 순위를 합치는 방식을 설명합니다

**배경**: `EnsembleRetriever`를 쓰려면 식을 직접 계산할 필요는 없지만, 설정값이 무엇을 바꾸는지는 알아야 합니다. 교안 01의 3·4절을 떠올리며 말로 정리합니다.

**요구사항**:

- **점수 대신 등수**: BM25 점수와 Dense 거리를 그대로 더하지 않고 등수를 쓰는 이유를 쓰세요.
- **`c`와 `k`**: `EnsembleRetriever`의 `c=60`과 검색기의 `k=2`가 각각 무엇을 정하는지 구분해 쓰세요.
- **두 목록에 모두 나온 문서**: 한 검색기에서만 1위인 문서보다 두 검색기에서 모두 2위인 문서가 앞설 수 있는 이유를 쓰세요.

**예시**: 항목마다 2~3문장으로 씁니다. 숫자를 계산하지 말고 개념으로 설명합니다.

<details><summary>힌트</summary>

```text
접근방법:
- RRF가 무엇을 입력으로 받고 무엇을 더하는지부터 떠올립니다.

세부구현:
1. 두 검색기의 점수 척도가 어떻게 다른지 적습니다.
2. c는 RRF 식의 상수, k는 검색기의 반환 개수라는 점을 구분합니다.
3. 같은 문서가 여러 목록에 있으면 기여를 더한다는 점을 연결합니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*


## 8. BM25가 점수를 매기는 기준을 설명합니다

**배경**: BM25의 설정값 `k1`·`b`를 바꾸기 전에 각 요소가 무엇을 반영하는지 알아야 합니다. 교안 01의 1절(DF·TF·문서 길이)을 떠올리며 말로 정리합니다.

**요구사항**:

- **희귀도**: 여러 문서에 흔히 나오는 토큰과 한두 문서에만 나오는 토큰 가운데 어느 쪽이 점수에 더 크게 기여하는지와 그 이유를 쓰세요.
- **반복 횟수**: 같은 토큰이 한 문서에 여러 번 나올 때 점수가 반복 횟수에 비례해 계속 늘지 않는 이유와 `k1`의 역할을 쓰세요.
- **문서 길이**: 같은 횟수가 나와도 긴 문서의 기여가 줄어드는 이유와 `b`의 역할을 쓰세요.

**예시**: 항목마다 2~3문장으로 씁니다. 수식을 계산하지 말고 뜻으로 설명합니다.

<details><summary>힌트</summary>

```text
접근방법:
- DF(등장 문서 수), TF(문서 안 등장 횟수), 문서 길이가 각각 점수에 어떻게 들어가는지 떠올립니다.

세부구현:
1. 희귀한 토큰일수록 문서를 구별하는 정도가 크다는 점을 씁니다.
2. 반복이 늘어도 증가량이 줄어든다는 점과 k1을 연결합니다.
3. 긴 문서에는 토큰이 우연히 많이 나올 수 있다는 점과 b를 연결합니다.
```

</details>


*(여기에 판단과 근거를 서술하세요)*
